# Przegląd wydajności i precyzyjności modeli detekcji obiektów
---
Wszystkie modele objęte ninieszym opracowaniem wybrano pod kątem ich najlepszego przystosowania do pełnienia konkretnej roli. 
Każdy z modeli objętych analizom posiada permisywną licencję umożliwiającą dostrojenie i komercyjne wykorzystanie modelu - tym samym umożliwiając użycie dostrojonych pochodnych w środowiskach komercyjnych.

Środowisko wykonawcze oparto na darmowej wersji Google Colab na architekturze ***GPU T4*** języku ***Python3.14*** oraz bibliotekach przeznaczonych do rozwoju uczenia maszynowego ***PyTorch***, ***TensorFlow*** oraz ***JAX***.

---

### Zestawienie modeli detekcji wziętych pod uwagę

| Nazwa modelu | Architektura modelu | Właściciel praw | Licencja | Dataset treningowy | AP (wg. oficjalnego źródła) |
| -- | -- | -- | -- | -- | -- |
| fasterrcnn_resnet50_fpn | CNN (ResNet-50 + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 37.0 |
| fasterrcnn_mobilenet_v3_large_fpn | CNN (MobileNetV3-Large + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 32.8 |
| retinanet_resnet50_fpn | CNN (ResNet-50 + FPN, RetinaNet) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 36.4 |
| retinanet_resnet50_fpn_v2 | CNN (ResNet-50 + FPN, RetinaNet v2) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 41.5 |
| fcos_resnet50_fpn | CNN (ResNet-50 + FPN, anchor-free FCOS) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 39.2 |
| ssd300_vgg16 | CNN (VGG16, SSD300) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 25.1 |
| ssdlite320_mobilenet_v3_large | CNN (MobileNetV3-Large, SSDLite320) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 21.3 |
| facebook/detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Facebook / Meta AI| Apache-2.0 | COCO 2017 | 42.0 |
| microsoft/conditional-detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Microsoft | Apache-2.0 | COCO 2017 | — |
| SenseTime/deformable-detr | Hybrydowa (CNN backbone + Transformer) | SenseTime | Apache-2.0 | COCO 2017 | — |
| PekingU/rtdetr_r18vd | Hybrydowa (CNN backbone + Transformer) | Peking University (PekingU) | Apache-2.0 | COCO train2017 | 46.5 |

In [34]:
# Import all neccessary libraries
import os
import sys
import time
import psutil
import torch
import numpy as np
import PIL.Image as Image
from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

# silence huggingface logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY_LEVEL"] = "error"
disable_progress_bars()

# silence transformers logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY_LEVEL"] = "error"
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()


# import pyvision models
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, fasterrcnn_mobilenet_v3_large_fpn,
    retinanet_resnet50_fpn, retinanet_resnet50_fpn_v2,
    fcos_resnet50_fpn, ssd300_vgg16, ssdlite320_mobilenet_v3_large
)
from torchvision.models import list_models, get_model, get_model_weights
# import transformers utils
from transformers import AutoModelForObjectDetection, AutoImageProcessor

# silence library warnings




In [35]:
# define constant model list
MODELS = [
    "fasterrcnn_resnet50_fpn",
    "fasterrcnn_mobilenet_v3_large_fpn",
    "retinanet_resnet50_fpn",
    "retinanet_resnet50_fpn_v2",
    "fcos_resnet50_fpn",
    "ssd300_vgg16",
    "ssdlite320_mobilenet_v3_large",
    "facebook/detr-resnet-50",
    "microsoft/conditional-detr-resnet-50",
    "SenseTime/deformable-detr",
    "PekingU/rtdetr_r18vd"
]

# define test environment

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEBUG: Using device: {device}")

DEBUG: Using device: cuda


In [ ]:
# loading model function
def load_model(model_name, device):
    tv_models = set(list_models())
    if model_name in tv_models:
        # print(f"DEBUG: Loading torchvision model: {model_name}")
        weights = get_model_weights(model_name).DEFAULT
        model = get_model(model_name, weights=weights).to(device).eval()
        preprocessing = weights.transforms()

        meta = {
            "model_name": model_name,
            "type": "torchvision",
            "weights": str(weights),
            "categories": weights.meta.get("categories"),
        }
        return model, preprocessing, meta
    
    # print(f"DEBUG: Loading transformers model: {model_name}")
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForObjectDetection.from_pretrained(model_name).assign(True).to(device).eval()
    meta = {
        "model_name": model_name,
        "type": "transformers",
        "weights": "from pretrained",
        "categories": model.config.id2label,
    }
    return model, processor, meta

# look for meta parameters in the model metadata
# this is to check if the metadata contains the necessary information for benchmarking
# if not, we can log a warning and skip the model
# this is important because some models might not have the necessary information for benchmarking, such as the categories or the input size
# we can log a warning and skip the model if the necessary information is not present in the metadata
def check_meta_parameters(model, model_name):
    meta_params = [
        name for name, param in model.named_parameters()
        if getattr(param, "is_meta", False)
    ]

    meta_buffers = [
        name for name, buffer in model.named_buffers()
        if getattr(buffer, "is_meta", False)
    ]

    if meta_params or meta_buffers:
        print(f"[WARNING] {model_name}: model still has meta tensors")
        print(f"Meta parameters: {len(meta_params)}")
        print(f"Meta buffers: {len(meta_buffers)}")
        print("First examples:", meta_params[:10])
        return False

    print(f"[OK] {model_name}: no meta tensors detected")
    return True 

In [42]:
# loding model and logging metadata

loaded_models = []

for n, model_name in enumerate(MODELS, start=1):
    try:
        model, processor, metadata = load_model(model_name, device)
        print(f"[{n}/{len(MODELS)}] Loaded {model_name}")
        check_meta_parameters(model, model_name)
        loaded_models.append(model_name)
    except Exception as e:
        print(f"ERROR while loading {model_name}: {e}")

ERROR while loading fasterrcnn_resnet50_fpn: 'FasterRCNN' object has no attribute 'assign'
ERROR while loading fasterrcnn_mobilenet_v3_large_fpn: 'FasterRCNN' object has no attribute 'assign'
ERROR while loading retinanet_resnet50_fpn: 'RetinaNet' object has no attribute 'assign'
ERROR while loading retinanet_resnet50_fpn_v2: 'RetinaNet' object has no attribute 'assign'
ERROR while loading fcos_resnet50_fpn: 'FCOS' object has no attribute 'assign'
ERROR while loading ssd300_vgg16: 'SSD' object has no attribute 'assign'
ERROR while loading ssdlite320_mobilenet_v3_large: 'SSD' object has no attribute 'assign'


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2586: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, whi

[8/11] Loaded facebook/detr-resnet-50
[OK] facebook/detr-resnet-50: no meta tensors detected
[9/11] Loaded microsoft/conditional-detr-resnet-50
[OK] microsoft/conditional-detr-resnet-50: no meta tensors detected
[10/11] Loaded SenseTime/deformable-detr
[OK] SenseTime/deformable-detr: no meta tensors detected
[11/11] Loaded PekingU/rtdetr_r18vd
[OK] PekingU/rtdetr_r18vd: no meta tensors detected
